In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

# 1. Binary Classification Random Forest

In [3]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

smote = SMOTE(random_state=42)
X_train,y_train = smote.fit_resample(X_train,y_train)

y_train[y_train == 1].shape,y_train[y_train == 1].shape

((699,), (699,))

In [4]:
params = {
    'n_estimators': [int(x) for x in np.linspace(start=20, stop=50, num=5)], # Jumlah pohon diperkecil rentangnya agar komputasi tidak mubazir saat memotong pohon
    'criterion': ['gini', 'entropy'],# Kriteria pembagian tetap sama
    'max_depth': [7, 10, 12,15], # DIKETATKAN: Hapus None, fokus pada kedalaman dangkal hingga sedang (7 sampai 15 saja)
    'min_samples_split': [15, 20, 25], # DIKETATKAN: Naikkan angka minimal sampel untuk split agar pohon tidak gampang bercabang
    'min_samples_leaf': [12, 16,20,29], # DIKETATKAN: Naikkan angka minimal di daun agar model menggeneralisasi lebih baik
    'max_features': ['sqrt', 'log2'], # Jumlah fitur acak tetap menggunakan subset terkecil
    'bootstrap': [True], # DIKETATKAN: Wajib True karena False memicu overfitting (memaksa pohon pakai semua data)
    'class_weight': ['balanced', None]# Pengatur bobot kelas disederhanakan
}
rdf = RandomForestClassifier()
random_search  = RandomizedSearchCV(estimator=rdf,param_distributions=params,n_iter=20,
                                    cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_rdf = random_search.best_estimator_

sfs_dct = SequentialFeatureSelector(estimator=best_model_rdf,n_features_to_select=5,direction='forward')
sfs_dct.fit(X_train,y_train)

X_train_selected = sfs_dct.transform(X_train)
X_test_selected = sfs_dct.transform(X_test)

fitur_terpilih = X_train.columns[sfs_dct.get_support()]
best_model_rdf.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')


Hyperparameter Terbaik: {'n_estimators': 50, 'min_samples_split': 20, 'min_samples_leaf': 20, 'max_features': 'log2', 'max_depth': 7, 'criterion': 'gini', 'class_weight': 'balanced', 'bootstrap': True}

Akurasi Validasi Terbaik: 81.19

Fitur Terbaik Yang Terpilih:
['Usia', 'Tipe_Nyeri_Dada', 'Detak_Jantung_Max', 'Oldpeak_ST', 'BMI']


In [5]:
test_accuracy = best_model_rdf.score(X_test_selected,y_test)
train_accruacy = best_model_rdf.score(X_train_selected,y_train)

y_pred_test = best_model_rdf.predict(X_test_selected)
y_prob_test = best_model_rdf.predict_proba(X_test_selected)

y_pred_train = best_model_rdf.predict(X_train_selected)
y_prob_train = best_model_rdf.predict_proba(X_train_selected)

print(f'\nAkurasi Pada Data Test: {test_accuracy*100:.2f}')
print(f'\nAkurasi Pada Data Train: {train_accruacy*100:.2f}')


Akurasi Pada Data Test: 73.67

Akurasi Pada Data Train: 80.33


In [6]:
def evaluate_model(pred,test,prob,evaluate,model_name='Decision Tree'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [7]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [8]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
           Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss                  Status
0  Decision Tree  Training  0.803290   0.801136  0.806867  0.803991  0.888948  0.441370  Overfitting (gap:6.66)
1  Decision Tree      Test  0.736667   0.798780  0.740113  0.768328  0.818244  0.526187  Overfitting (gap:6.66)




In [9]:
from sklearn.preprocessing import StandardScaler
feauter_numerik = ['Usia','Tekanan_Darah_Rest','Kolesterol','Detak_Jantung_Max','Oldpeak_ST','BMI']
data_5_pasien = {
    'Usia': [63, 45, 56, 38, 50],
    'Jenis_Kelamin': [3, 1, 0, 1, 0],
    'Tipe_Nyeri_Dada': [-0.3, 1.20, 0.08, -1.05, -0.48],
    'Tekanan_Darah_Rest': [140, 130, 122, 110, 120],
    'Kolesterol': [69, 100, 122, 115, 190],
    'Gula_Darah_Puasa': [1, 0, 0, 0, 1],
    'EKG_Rest': [1, 2, 1, 1, 1],
    'Detak_Jantung_Max': [-0.98, 0.90, -1.05, 0.29, -1.58],
    'Angina_Olahraga': [44, 60, 56, 75, 14],
    'Oldpeak_ST': [19, 59, 73, 76, 18],
    'Kemiringan_ST': [1.28, 2.0, 1.5, 1.5, 2.0],
    'Jumlah_Pembuluh_Darah': [46, 60, 4, 0, 3],
    'Thalassemia': [81, 90, 1, 4, 1],
    'BMI': [10.77, 81.24, 24.5, 28.1, 29.3]
}
scaler = StandardScaler()
data_pasien= pd.DataFrame(data_5_pasien)
data_pasien[feauter_numerik] = scaler.fit_transform(data_pasien[feauter_numerik])
target_asli_pasien = [1, 0, 1, 0, 1] 

data_pasien_baru_x = data_pasien[fitur_terpilih]
data_pasien_baru_y = target_asli_pasien
data_pasien_baru_x

,Usia,Tipe_Nyeri_Dada,Detak_Jantung_Max,Oldpeak_ST,BMI
0,1.458427,-0.30,-0.535966,-1.173811,-0.994417
1,-0.625040,1.20,1.495518,0.391270,1.923981
2,0.648190,0.08,-0.611606,0.939049,-0.425812
3,-1.435277,-1.05,0.836366,1.056430,-0.276724
4,-0.046299,-0.48,-1.184312,-1.212938,-0.227028


In [10]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
# data_pasien_baru = df_x.sample(2, random_state=10)
data_pasien_baru = data_pasien[fitur_terpilih]
prediksi_pasien = best_model_rdf.predict(data_pasien_baru)
probabilitas_pasien = best_model_rdf.predict_proba(data_pasien_baru)

hasil_df = data_pasien_baru.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_rdf.score(data_pasien_baru_x, data_pasien_baru_y)
prediksi_pasien = best_model_rdf.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_rdf.predict_proba(data_pasien_baru_x)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))

hasil_df[['Peluang Aman(%)','Peluang Terkena (%)','Prediksi Serangan Jantung']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 80.00%

Tabel Skor Evaluasi Lengkap Data Baru:
        Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Decision Tree Data_Baru       0.8       0.75     1.0  0.857143      1.0  0.362564


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,25.255424,74.744576,1
1,77.922173,22.077827,0
2,3.518767,96.481233,1
3,47.307694,52.692306,1
4,38.611633,61.388367,1


# 2.Multiclass Classification Random Forest